# Análisis de tickets: primer diagnóstico del dataset

Objetivo de este primer paso: **ver qué contiene `tickets`** y detectar anomalías básicas antes de cruzarlo con reservas, pagos y cambios de sistema.

Trabajamos sobre la copia `data/work/tickets_trabajo.csv`. El original `data/raw/tickets.csv` no se modifica.

In [1]:
import pandas as pd

RUTA = "../data/work/tickets_trabajo.csv"

tickets = pd.read_csv(RUTA, dtype=str)
for col in ["issued_at", "voided_at"]:
    tickets[col] = pd.to_datetime(tickets[col], utc=True, errors="coerce")

print("Filas, columnas:", tickets.shape)
tickets.head()

Filas, columnas: (176562, 8)


,ticket_id,reservation_id,payment_id,concert_id,seat_id,issued_at,voided_at,issuer_version
0,TKT-0100396,RES-0107474,PAY-0101433,CON-04,S18-R18-B23,2026-10-16 16:41:57+00:00,NaT,4.8.0
1,TKT-0137553,RES-0147254,PAY-0138919,CON-05,S24-R10-B19,2026-10-23 12:42:15+00:00,NaT,7.3.1
2,TKT-0151929,RES-0162702,PAY-0153410,CON-06,S04-R35-B11,2026-10-23 13:25:18+00:00,NaT,4.8.0
3,TKT-0115245,RES-0123222,PAY-0116308,CON-05,S01-R12-B17,2026-10-23 16:46:45+00:00,NaT,7.3.1
4,TKT-0061234,RES-0065679,PAY-0061997,CON-03,S04-R35-B19,2026-10-16 13:00:36+00:00,NaT,7.3.1


## 1. Estructura y calidad básica

Comprobamos nulos, unicidad de claves, formato de identificadores y rango de fechas.

In [2]:
print("Nulos por columna:")
print(tickets.isna().sum(), "\n")

print("ticket_id únicos:      ", tickets.ticket_id.nunique())
print("reservation_id únicos: ", tickets.reservation_id.nunique())
print("payment_id únicos:     ", tickets.payment_id.nunique())
print("Filas duplicadas exactas:", tickets.duplicated().sum(), "\n")

formatos = {
    "ticket_id": r"^TKT-\d{7}$",
    "reservation_id": r"^RES-\d{7}$",
    "payment_id": r"^PAY-\d{7}$",
    "concert_id": r"^CON-\d{2}$",
    "seat_id": r"^S\d{2}-R\d{2}-B\d{2}$",
}
for col, patron in formatos.items():
    print(f"Formato incorrecto en {col}:", (~tickets[col].str.match(patron, na=False)).sum())

print("\nRango issued_at:", tickets.issued_at.min(), "→", tickets.issued_at.max())
print("Rango voided_at: ", tickets.voided_at.min(), "→", tickets.voided_at.max())

Nulos por columna:
ticket_id              0
reservation_id         0
payment_id             0
concert_id             0
seat_id                0
issued_at              0
voided_at         172997
issuer_version         0
dtype: int64 

ticket_id únicos:       176562
reservation_id únicos:  176562
payment_id únicos:      176562


Filas duplicadas exactas:

 0 

Formato incorrecto en ticket_id: 0
Formato incorrecto en reservation_id: 0
Formato incorrecto en payment_id: 0
Formato incorrecto en concert_id: 0
Formato incorrecto en seat_id: 0

Rango issued_at: 2026-10-02 10:04:27+00:00 → 2026-10-26 17:46:44+00:00


Rango voided_at:  2026-10-03 10:05:27+00:00 → 2026-10-26 17:25:54+00:00


## 2. Qué contiene: conciertos, versiones y anulaciones

- Cada fila es una entrada emitida (una reserva, un pago, un asiento).
- `voided_at` vacío = entrada vigente. Cuando tiene fecha, la entrada ha sido anulada.

In [3]:
tickets["vigente"] = tickets.voided_at.isna()

resumen_conciertos = (
    tickets.groupby("concert_id")
    .agg(
        emitidas=("ticket_id", "size"),
        vigentes=("vigente", "sum"),
        anuladas=("vigente", lambda s: (~s).sum()),
        primera_emision=("issued_at", "min"),
        ultima_emision=("issued_at", "max"),
    )
)
resumen_conciertos["vigentes"] = resumen_conciertos["vigentes"].astype(int)
resumen_conciertos

,emitidas,vigentes,anuladas,primera_emision,ultima_emision
concert_id,,,,,
CON-01,27108,26591,517,2026-10-02 10:04:27+00:00,2026-10-05 17:13:48+00:00
CON-02,30375,29769,606,2026-10-03 10:04:27+00:00,2026-10-06 18:19:24+00:00
CON-03,25434,24911,523,2026-10-16 10:04:27+00:00,2026-10-19 17:20:04+00:00
CON-04,32055,31403,652,2026-10-16 10:04:27+00:00,2026-10-19 16:55:04+00:00
CON-05,33058,32365,693,2026-10-23 10:04:27+00:00,2026-10-26 17:46:44+00:00
CON-06,28532,27958,574,2026-10-23 10:04:27+00:00,2026-10-26 17:22:41+00:00


In [4]:
pd.crosstab(tickets.concert_id, tickets.issuer_version, margins=True)

issuer_version,4.7.2,4.8.0,7.3.1,All
concert_id,,,,
CON-01,0,0,27108,27108
CON-02,30375,0,0,30375
CON-03,0,0,25434,25434
CON-04,5143,26912,0,32055
CON-05,0,0,33058,33058
CON-06,0,28532,0,28532
All,35518,55444,85600,176562


In [5]:
# Anulaciones anteriores a la emisión (incoherencia temporal)
print("Anuladas antes de emitirse:", (tickets.voided_at < tickets.issued_at).sum())

Anuladas antes de emitirse: 0


## 3. Localidades con más de una entrada vigente

Regla del servicio: una localidad (`concert_id` + `seat_id`) no puede tener dos entradas vigentes a la vez. Dos entradas sucesivas, con la primera anulada, sí son legítimas.

Aquí buscamos grupos de **entradas vigentes** que comparten localidad.

In [6]:
vigentes = tickets[tickets.vigente]

duplicadas = vigentes[vigentes.duplicated(["concert_id", "seat_id"], keep=False)]
grupos = duplicadas.groupby(["concert_id", "seat_id"])

print("Localidades con 2+ entradas vigentes:", grupos.ngroups)
print("Entradas implicadas:                 ", len(duplicadas))
print("\nPor concierto e issuer_version:")
print(duplicadas.groupby(["concert_id", "issuer_version"]).size().unstack(fill_value=0))

Localidades con 2+ entradas vigentes: 1195
Entradas implicadas:                  2390

Por concierto e issuer_version:
issuer_version  4.8.0
concert_id           
CON-04           1120
CON-06           1270


In [7]:
# Separación temporal entre las dos entradas de cada localidad
ventana = grupos.issued_at.agg(["min", "max"])
ventana["separacion_s"] = (ventana["max"] - ventana["min"]).dt.total_seconds()

print(ventana.separacion_s.describe().round(1))
print("\nLocalidades con emisiones separadas por menos de 10 min:", (ventana.separacion_s < 600).sum(), "de", len(ventana))

count    1195.0
mean       89.6
std        55.5
min         1.0
25%        43.0
50%        85.0
75%       131.0
max       233.0
Name: separacion_s, dtype: float64

Localidades con emisiones separadas por menos de 10 min: 1195 de 1195


In [8]:
# Ejemplo de localidades afectadas
duplicadas.sort_values(["concert_id", "seat_id", "issued_at"]).head(10)

,ticket_id,reservation_id,payment_id,concert_id,seat_id,issued_at,voided_at,issuer_version,vigente
98391,TKT-0082919,RES-0088921,PAY-0083931,CON-04,S01-R01-B02,2026-10-16 14:11:25+00:00,NaT,4.8.0,True
4411,TKT-0082920,RES-0088922,PAY-0083932,CON-04,S01-R01-B02,2026-10-16 14:11:47+00:00,NaT,4.8.0,True
127456,TKT-0082951,RES-0088954,PAY-0083963,CON-04,S01-R02-B07,2026-10-16 16:49:25+00:00,NaT,4.8.0,True
57151,TKT-0082950,RES-0088953,PAY-0083962,CON-04,S01-R02-B07,2026-10-16 16:52:41+00:00,NaT,4.8.0,True
171293,TKT-0083047,RES-0089054,PAY-0084059,CON-04,S01-R06-B04,2026-10-16 14:16:20+00:00,NaT,4.8.0,True
79387,TKT-0083048,RES-0089055,PAY-0084060,CON-04,S01-R06-B04,2026-10-16 14:16:59+00:00,NaT,4.8.0,True
19170,TKT-0083110,RES-0089121,PAY-0084122,CON-04,S01-R08-B16,2026-10-16 13:29:39+00:00,NaT,4.8.0,True
133805,TKT-0083109,RES-0089120,PAY-0084121,CON-04,S01-R08-B16,2026-10-16 13:30:30+00:00,NaT,4.8.0,True
115122,TKT-0083121,RES-0089132,PAY-0084133,CON-04,S01-R09-B01,2026-10-16 12:33:27+00:00,NaT,4.8.0,True
111889,TKT-0083120,RES-0089131,PAY-0084132,CON-04,S01-R09-B01,2026-10-16 12:33:40+00:00,NaT,4.8.0,True


## 4. Hallazgos preliminares y límites

**Lo que muestran los datos de `tickets`:**
- Hay localidades con dos entradas vigentes a la vez. En este dataset todas corresponden a una misma versión del emisor, y se concentran en dos conciertos.
- Las anulaciones son coherentes en tiempo (ninguna es anterior a su emisión).

**Lo que todavía no podemos afirmar solo con este archivo:**
- Que las entradas se emitieran sobre reservas caducadas. Para eso hay que cruzar con `reservations.csv` (`expires_at`).
- Que hubo cobro sin entrada o viceversa. Requiere `payment_events.csv`.
- Que un cambio de configuración sea la causa. Solo hay coincidencia temporal y de versión; hay que comparar con operaciones correctas del mismo proveedor y versión antes y después del cambio.

Siguiente paso: cruzar `tickets` con `reservations` y `payment_events` por `reservation_id` y `payment_id`.

## 5. Desglose de CON-04 por versión del emisor

CON-04 (EntradaLink) aparece con dos versiones del emisor: 4.7.2 y 4.8.0. Comprobamos si el cambio de versión coincide con el momento de la anomalía y si la mezcla es legítima.

In [9]:
c04 = tickets[tickets.concert_id == "CON-04"].copy()

resumen = (
    c04.groupby("issuer_version")
    .agg(
        emitidas=("ticket_id", "size"),
        vigentes=("vigente", "sum"),
        primera_emision=("issued_at", "min"),
        ultima_emision=("issued_at", "max"),
    )
)
resumen["anuladas"] = resumen.emitidas - resumen.vigentes
resumen

,emitidas,vigentes,primera_emision,ultima_emision,anuladas
issuer_version,,,,,
4.7.2,5143,5023,2026-10-16 10:04:27+00:00,2026-10-16 10:20:01+00:00,120
4.8.0,26912,26380,2026-10-16 10:20:02+00:00,2026-10-19 16:55:04+00:00,532


### Corte de versión

El cambio de 4.7.2 a 4.8.0 debería verse como un corte limpio en el tiempo. Según `system_changes.csv`, CHG-004 (EL, 4.8.0) entra en vigor el 16/10/2026 a las 10:20Z.

In [10]:
ultima_472 = c04.loc[c04.issuer_version == "4.7.2", "issued_at"].max()
primera_480 = c04.loc[c04.issuer_version == "4.8.0", "issued_at"].min()
print("Última emisión 4.7.2:", ultima_472)
print("Primera emisión 4.8.0:", primera_480)
print("Emisiones 4.8.0 anteriores a la primera 4.8.0 (sanity):",
      (c04.issuer_version.eq("4.8.0") & (c04.issued_at < primera_480)).sum())
print("Emisiones 4.8.0 por día:")
print(c04[c04.issuer_version == "4.8.0"].groupby(c04.issued_at.dt.date).size())

Última emisión 4.7.2: 2026-10-16 10:20:01+00:00
Primera emisión 4.8.0: 2026-10-16 10:20:02+00:00
Emisiones 4.8.0 anteriores a la primera 4.8.0 (sanity): 0
Emisiones 4.8.0 por día:
issued_at
2026-10-16    26260
2026-10-17      146
2026-10-18      310
2026-10-19      196
dtype: int64


### Duplicados vigentes por versión

Solo nos interesan las entradas vigentes que comparten localidad dentro de CON-04.

In [11]:
v04 = c04[c04.vigente]
dup04 = v04[v04.duplicated(["seat_id"], keep=False)]

print("Entradas vigentes duplicadas por versión:")
print(dup04.groupby("issuer_version").size())
print("\nLocalidades afectadas por versión:")
print(dup04.groupby("issuer_version").seat_id.nunique())
print("\nEntradas vigentes 4.8.0 totales:", (v04.issuer_version == "4.8.0").sum())

Entradas vigentes duplicadas por versión:
issuer_version
4.8.0    1120
dtype: int64

Localidades afectadas por versión:
issuer_version
4.8.0    560
Name: seat_id, dtype: int64

Entradas vigentes 4.8.0 totales: 26380


### Localidades que aparecen en las dos versiones

Una localidad vendida con 4.7.2 y después con 4.8.0 es legítima si la entrada 4.7.2 fue anulada antes de la nueva emisión.

In [12]:
ambas = set(c04.loc[c04.issuer_version == "4.7.2", "seat_id"]) & set(c04.loc[c04.issuer_version == "4.8.0", "seat_id"])

legitimas, conflictivas = 0, []
for seat, g in c04[c04.seat_id.isin(ambas)].groupby("seat_id"):
    a = g[g.issuer_version == "4.7.2"]
    b = g[g.issuer_version == "4.8.0"]
    if a.voided_at.notna().all() and a.voided_at.max() <= b.issued_at.min():
        legitimas += 1
    else:
        conflictivas.append(seat)

print("Localidades en ambas versiones:", len(ambas))
print("  Reemisión legítima (4.7.2 anulada antes):", legitimas)
print("  Sin anulación previa:", len(conflictivas))

Localidades en ambas versiones: 120
  Reemisión legítima (4.7.2 anulada antes): 120
  Sin anulación previa: 0


**Conclusión del desglose de CON-04**

- La versión 4.7.2 emite solo hasta las 10:20:01Z del 16/10. La 4.8.0 empieza a las 10:20:02Z, coincidiendo con CHG-004. No hay emisiones 4.8.0 antes del cambio.
- Todas las entradas vigentes duplicadas de CON-04 (1.120 entradas en 560 localidades) están en 4.8.0. Son el 4,25 % de las entradas vigentes 4.8.0 del concierto.
- Las 120 localidades presentes en ambas versiones son reventas legítimas: la 4.7.2 se anuló antes de la nueva emisión.
- Por tanto, la mezcla de versiones en CON-04 no es en sí una incidencia. La incidencia está en las emisiones 4.8.0 con localidad ya vigente.

Esto sigue sin verificar la caducidad de las reservas, que requiere cruzar con `reservations.csv`.